# Apex Retail Intelligence — Phase 6: Business Reporting (KPI Generation)

**Deliverable:** PySpark logic and resulting outputs rendered within Databricks notebook.

**Constraint:** No external dashboards (Power BI, Tableau). All KPIs computed via PySpark DataFrames / Spark SQL.

---

### Required KPIs (per assignment spec)

| # | KPI Name | Definition |
| --- | --- | --- |
| 1 | **Net Margin by Region** | Total gross revenue minus discounts, grouped by store region |
| 2 | **Average Order Value (AOV) by Promotion** | Which promotion types drive the highest average cart values |
| 3 | **Demographic Churn Heatmap** | Customer churn rates split by state and loyalty programme membership |
| 4 | **Product Quality Index** | Which product categories suffer the highest return rates |
| 5 | **Store Traffic by Hour** | Busiest transaction hours and days of the week |

---

### Data Source

All queries read from **`apex_retail.GOLD_tables`** star schema (Unity Catalog).

**Note:** Source data uses simplified sample columns. Approximations are documented per KPI.

In [0]:
from pyspark.sql import functions as F

GOLD_CATALOG = "apex_retail"
GOLD_SCHEMA = "GOLD_tables"

# Load Gold layer tables
fact_sales = spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.fact_sales")
dim_customer = spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_customer")
dim_product = spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_product")
dim_date = spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_date")

print(f"fact_sales:    {fact_sales.count()} rows")
print(f"dim_customer:  {dim_customer.count()} rows")
print(f"dim_product:   {dim_product.count()} rows")
print(f"dim_date:      {dim_date.count()} rows")

fact_sales:    120 rows
dim_customer:  60 rows
dim_product:   35 rows
dim_date:      120 rows


In [0]:
%sql
-- KPI 1: Net Margin by Region
-- Definition: Total gross revenue minus discounts, grouped by store region.
-- Note: discount_applied not in source; net_margin = revenue (discount = 0)

SELECT
  c.state                             AS region,
  c.city,
  COUNT(f.sale_id)                    AS total_orders,
  SUM(f.quantity)                     AS total_units,
  ROUND(SUM(f.total_amount), 2)       AS gross_revenue,
  0.00                                AS total_discount,
  ROUND(SUM(f.total_amount), 2)       AS net_margin,
  ROUND(SUM(f.total_amount) / SUM(SUM(f.total_amount)) OVER () * 100, 2) AS margin_share_pct
FROM apex_retail.GOLD_tables.fact_sales f
LEFT JOIN apex_retail.GOLD_tables.dim_customer c
  ON f.customer_sk = c.customer_sk AND c.is_active = TRUE
GROUP BY c.state, c.city
ORDER BY net_margin DESC

region,city,total_orders,total_units,gross_revenue,total_discount,net_margin,margin_share_pct
TX,Houston,28,81,7771.59,0.00,7771.59,26.72
NY,New York,26,71,6960.26,0.00,6960.26,23.93
TX,Dallas,27,70,5328.05,0.00,5328.05,18.32
IL,Chicago,18,51,4458.34,0.00,4458.34,15.33
AZ,Phoenix,19,59,3868.79,0.00,3868.79,13.30
MA,Boston,1,2,395.35,0.00,395.35,1.36
CO,Denver,1,3,302.32,0.00,302.32,1.04


In [0]:
%sql
-- KPI 2: Average Order Value (AOV) by Promotion
-- Definition: Identify which promotion types drive the highest average cart values.
-- Joins fact_sales to dim_promotion on promotion_sk.

SELECT
  pr.promotion_id,
  pr.promotion_type,
  COUNT(f.sale_id)                AS total_orders,
  SUM(f.quantity)                 AS total_units,
  ROUND(AVG(f.total_amount), 2)   AS avg_order_value,
  ROUND(SUM(f.total_amount), 2)   AS total_revenue,
  ROUND(MIN(f.total_amount), 2)   AS min_order_value,
  ROUND(MAX(f.total_amount), 2)   AS max_order_value
FROM apex_retail.GOLD_tables.fact_sales f
LEFT JOIN apex_retail.GOLD_tables.dim_promotion pr
  ON f.promotion_sk = pr.promotion_sk
GROUP BY pr.promotion_id, pr.promotion_type
ORDER BY avg_order_value DESC

promotion_id,promotion_type,total_orders,total_units,avg_order_value,total_revenue,min_order_value,max_order_value
NONE,No Promotion,120,337,242.37,29084.70,11.06,492.28


In [0]:
%sql
-- KPI 3: Demographic Churn Heatmap
-- Definition: Customer churn rates split by state and loyalty programme membership.
-- Approximation: churn = no purchase in last 30 days from max date.
-- Loyalty tier = signup tenure (Early Adopter >60d, Regular 30-60d, New <30d).

WITH max_dt AS (
  SELECT MAX(sale_date) AS ref_date FROM apex_retail.GOLD_tables.fact_sales
),
customer_recency AS (
  SELECT
    c.customer_sk,
    c.state,
    c.signup_date,
    MAX(f.sale_date) AS last_purchase,
    DATEDIFF((SELECT ref_date FROM max_dt), MAX(f.sale_date)) AS days_since_purchase,
    CASE
      WHEN DATEDIFF((SELECT ref_date FROM max_dt), c.signup_date) > 60 THEN 'Early Adopter'
      WHEN DATEDIFF((SELECT ref_date FROM max_dt), c.signup_date) > 30 THEN 'Regular'
      ELSE 'New'
    END AS loyalty_tier
  FROM apex_retail.GOLD_tables.fact_sales f
  INNER JOIN apex_retail.GOLD_tables.dim_customer c
    ON f.customer_sk = c.customer_sk AND c.is_active = TRUE
  GROUP BY c.customer_sk, c.state, c.signup_date
)
SELECT
  state,
  loyalty_tier,
  COUNT(*) AS total_customers,
  SUM(CASE WHEN days_since_purchase > 30 THEN 1 ELSE 0 END) AS churned_customers,
  ROUND(SUM(CASE WHEN days_since_purchase > 30 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM customer_recency
GROUP BY state, loyalty_tier
ORDER BY churn_rate_pct DESC

state,loyalty_tier,total_customers,churned_customers,churn_rate_pct
AZ,Early Adopter,8,7,87.50
IL,Early Adopter,8,7,87.50
TX,Early Adopter,19,12,63.16
NY,Early Adopter,7,3,42.86
MA,Regular,1,0,0.00
CO,Regular,1,0,0.00


In [0]:
%sql
-- KPI 4: Product Quality Index
-- Definition: Which product categories suffer the highest return rates.
-- Approximation: No return_rate in source. Proxy = quality index using
-- AOV / (StdDev + 1) * 10. Lower index = higher price variance = quality concern.

WITH product_stats AS (
  SELECT
    p.category,
    p.product_id,
    p.product_name,
    COUNT(f.sale_id)              AS total_orders,
    SUM(f.quantity)               AS total_units_sold,
    ROUND(AVG(f.total_amount), 2) AS avg_order_value,
    ROUND(STDDEV(f.total_amount), 2) AS order_value_stddev,
    0.00                          AS return_rate
  FROM apex_retail.GOLD_tables.fact_sales f
  LEFT JOIN apex_retail.GOLD_tables.dim_product p ON f.product_sk = p.product_sk
  GROUP BY p.category, p.product_id, p.product_name
)
SELECT
  *,
  ROUND(avg_order_value / (COALESCE(order_value_stddev, 0) + 1) * 10, 2) AS quality_index
FROM product_stats
ORDER BY quality_index ASC

category,product_id,product_name,total_orders,total_units_sold,avg_order_value,order_value_stddev,return_rate,quality_index
Electronics,P0006,Product_G6,2,4,180.72,234.98,0.00,7.66
Electronics,P0011,Product_L11,5,9,151.66,166.5,0.00,9.05
Home,P0018,Product_S18,5,18,96.36,100.84,0.00,9.46
Clothing,P0027,Product_B27,4,14,252.60,244.35,0.00,10.3
Food,P0010,Product_K10,3,7,153.18,133.71,0.00,11.37
Food,P0030,Product_E30,3,10,187.31,153.57,0.00,12.12
Sports,P0019,Product_T19,3,8,289.60,231.14,0.00,12.48
Electronics,P0026,Product_A26,2,2,202.52,158.18,0.00,12.72
Clothing,P0002,Product_C2,3,8,245.16,191.59,0.00,12.73
Food,P0005,Product_F5,4,9,282.15,215.03,0.00,13.06


In [0]:
%sql
-- KPI 5: Store Traffic by Hour
-- Definition: Identify the busiest transaction hours and days of the week.
-- Approximation: transaction_hour not in source. Showing day-of-week traffic.

SELECT
  d.day_of_week,
  CASE WHEN d.is_weekend THEN 'Weekend' ELSE 'Weekday' END AS day_type,
  COUNT(f.sale_id)                AS transaction_count,
  SUM(f.quantity)                 AS total_units,
  ROUND(SUM(f.total_amount), 2)   AS total_revenue,
  ROUND(AVG(f.total_amount), 2)   AS avg_transaction_value,
  CASE
    WHEN COUNT(f.sale_id) >= 3 THEN 'High'
    WHEN COUNT(f.sale_id) >= 2 THEN 'Medium'
    ELSE 'Low'
  END AS traffic_level
FROM apex_retail.GOLD_tables.fact_sales f
JOIN apex_retail.GOLD_tables.dim_date d ON f.date_sk = d.date_sk
GROUP BY d.day_of_week, d.is_weekend
ORDER BY transaction_count DESC, total_revenue DESC

day_of_week,day_type,transaction_count,total_units,total_revenue,avg_transaction_value,traffic_level
Wednesday,Weekday,18,53,4758.36,264.35,High
Sunday,Weekend,18,51,3563.67,197.98,High
Saturday,Weekend,17,41,4612.97,271.35,High
Friday,Weekday,17,54,4605.72,270.92,High
Tuesday,Weekday,17,47,4063.24,239.01,High
Thursday,Weekday,17,43,3421.79,201.28,High
Monday,Weekday,16,48,4058.95,253.68,High


In [0]:
# Phase 6 Executive Summary + Validation
total_revenue = fact_sales.agg(F.sum("total_amount")).collect()[0][0]
total_orders = fact_sales.count()
total_units = fact_sales.agg(F.sum("quantity")).collect()[0][0]
avg_order_val = fact_sales.agg(F.round(F.avg("total_amount"), 2)).collect()[0][0]
unique_customers = fact_sales.filter(F.col("customer_sk").isNotNull()).select("customer_sk").distinct().count()
unique_products = fact_sales.filter(F.col("product_sk").isNotNull()).select("product_sk").distinct().count()

print("="*60)
print("  PHASE 6: BUSINESS REPORTING (KPI GENERATION) COMPLETE")
print("="*60)
print(f"")
print(f"  Total Revenue:        ${total_revenue:,.2f}")
print(f"  Total Orders:          {total_orders:,}")
print(f"  Total Units Sold:      {total_units:,}")
print(f"  Avg Order Value:       ${avg_order_val:,.2f}")
print(f"  Unique Customers:      {unique_customers}")
print(f"  Unique Products:       {unique_products}")
print(f"")
print("-"*60)
print("  KPI 1: Net Margin by Region              Done")
print("  KPI 2: AOV by Promotion                  Done")
print("  KPI 3: Demographic Churn Heatmap         Done")
print("  KPI 4: Product Quality Index             Done")
print("  KPI 5: Store Traffic by Day of Week      Done")
print("-"*60)
print(f"")
print("  All KPIs computed using Spark SQL")
print("  Output rendered within Databricks notebook")
print("  Source: apex_retail.GOLD_tables (Star Schema)")
print("  Surrogate key joins: fact_sales -> dim_customer, dim_product, dim_date, dim_promotion")
print("="*60)

  PHASE 6: BUSINESS REPORTING (KPI GENERATION) COMPLETE

  Total Revenue:        $29,084.70
  Total Orders:          120
  Total Units Sold:      337
  Avg Order Value:       $242.37
  Unique Customers:      44
  Unique Products:       30

------------------------------------------------------------
  KPI 1: Net Margin by Region              Done
  KPI 2: AOV by Promotion                  Done
  KPI 3: Demographic Churn Heatmap         Done
  KPI 4: Product Quality Index             Done
  KPI 5: Store Traffic by Day of Week      Done
------------------------------------------------------------

  All KPIs computed using Spark SQL
  Output rendered within Databricks notebook
  Source: apex_retail.GOLD_tables (Star Schema)
  Surrogate key joins: fact_sales -> dim_customer, dim_product, dim_date, dim_promotion
